# Hybrid MPI-CUDA Distributed Stencil & Cross-Tier Cache-Line Coherency (Python)

4-rank 2 × 2 Cartesian MPI + CUDA stencil (GPU halo pack/unpack, 18 × 18 shared-memory tile, dual-stream overlap) coupled with an MPI-3 `Allocate_shared` false-sharing benchmark.

Run on a **T4 GPU** runtime with **Runtime → Run all**.

### Cell 1 — Hardware & MPI environment setup

In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!lscpu | grep -E "Model name|CPU\(s\)|cache" || true
!pip install -q mpi4py numba numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-f3b1c3b2-e7b4-4a25-2908-82dc3311f7e2, 7.5, 15360 MiB
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
L1d cache:                               32 KiB (1 instance)
L1i cache:                               32 KiB (1 instance)
L2 cache:                                1 MiB (1 instance)
L3 cache:                                38.5 MiB (1 instance)
NUMA node0 CPU(s):                       0,1


### Cell 2 — Solo student identity configuration

In [2]:
GITHUB_USERNAME = "230103278"
assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and len(GITHUB_USERNAME.strip()) > 0, (
    "Set GITHUB_USERNAME to your real GitHub handle before running."
)
print(f"Verified solo student handle: {GITHUB_USERNAME}")

Verified solo student handle: 230103278


### Cell 3 — Hybrid MPI-CUDA & shared-memory coherency engine (`%%writefile hybrid_mpi_cuda_cache.py`)

In [3]:
%%writefile hybrid_mpi_cuda_cache.py
import os
import re
import sys
import json
import math
import time
import hashlib
import platform
import subprocess
from datetime import datetime, timezone
from statistics import median
from types import SimpleNamespace

import numpy as np
from mpi4py import MPI
from numba import cuda, njit, float64, uint64

TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2  # 18x18 shared tile
GLOBAL_NX = 512
GLOBAL_NY = 512
STENCIL_STEPS = 300
SHM_ITERS = 40_000_000
LAMBDA_DAMP = 0.15

PROC_DIMS = [2, 2]
NY_LOCAL = GLOBAL_NY // PROC_DIMS[0]  # 256 owned rows per rank
NX_LOCAL = GLOBAL_NX // PROC_DIMS[1]  # 256 owned columns per rank
DX = 1.0 / (GLOBAL_NX + 1)  # 1/513: owned cells are global 1..512, Dirichlet walls sit at 0 and 513
DY = 1.0 / (GLOBAL_NY + 1)
CFL_SAFETY = 0.8  # dt = 0.8 * dx^2 / (4 * kappa_max)
REPORT_EVERY = 100
PACK_THREADS = 128

SHM_SLOTS = 64
SHM_STRIDES = (1, 8, 16)
SHM_REPEATS = 3  # median of 3 timed runs per stride
SHM_ALIGN = 128  # satisfies the 64-byte requirement and also aligns 128-byte adjacent-line pairs

# Segment order inside the 1D halo buffers. A strip sent towards direction d carries tag d,
# so the receiver expects the tag of the opposite direction.
UP, DOWN, LEFT, RIGHT = 0, 1, 2, 3
OPPOSITE = (DOWN, UP, RIGHT, LEFT)


def _abort_on_error(exc_type, exc, tb):
    sys.__excepthook__(exc_type, exc, tb)
    MPI.COMM_WORLD.Abort(1)  # one crashed rank must not leave the other three blocked in a collective


sys.excepthook = _abort_on_error


def get_gpu_uuid() -> str:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8",
    )
    return out.strip().splitlines()[0].strip()


def detect_cpu_line_size() -> int:
    path = "/sys/devices/system/cpu/cpu0/cache/index0/coherency_line_size"
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return int(f.read().strip())
    return 64


def read_cpu_model() -> str:
    try:
        with open("/proc/cpuinfo", encoding="utf-8") as f:
            for line in f:
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()


def physical_core_cpus():
    """First logical CPU of every physical core this process may run on."""
    if not hasattr(os, "sched_getaffinity"):
        return []
    chosen, seen = [], set()
    for cpu in sorted(os.sched_getaffinity(0)):
        try:
            with open(f"/sys/devices/system/cpu/cpu{cpu}/topology/thread_siblings_list", encoding="utf-8") as f:
                core = f.read().strip()
        except OSError:
            core = str(cpu)
        if core not in seen:
            seen.add(core)
            chosen.append(cpu)
    return chosen


# ==============================================================================
# TASK 1: Hybrid MPI Cartesian Grid + Node-Local MPI-3 Shared-Memory Window
# ==============================================================================
def init_hybrid_context(github_user: str) -> dict:
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Requires exactly 4 MPI ranks (2x2 grid), got {size}.")

    cuda.select_device(0)
    gpu_uuid = comm.bcast(get_gpu_uuid() if rank == 0 else None, root=0)
    cpu_sig = f"{platform.machine()}::{os.cpu_count()}"
    line_bytes = detect_cpu_line_size()

    digest = hashlib.sha256(f"{github_user.strip()}::{gpu_uuid}::{cpu_sig}".encode("utf-8")).hexdigest()
    instance_seed = int(digest[:16], 16)

    cart = comm.Create_cart(dims=PROC_DIMS, periods=[False, False], reorder=False)
    cart_rank = cart.Get_rank()
    nbr_up, nbr_down = cart.Shift(0, 1)  # axis 0 = rows (y)
    nbr_left, nbr_right = cart.Shift(1, 1)  # axis 1 = columns (x)

    node_comm = comm.Split_type(MPI.COMM_TYPE_SHARED)
    if node_comm.Get_size() != size:
        raise RuntimeError("All 4 ranks must share one node for the MPI-3 shared-memory window.")
    node_rank = node_comm.Get_rank()
    nbytes = SHM_SLOTS * 8 + SHM_ALIGN
    win = MPI.Win.Allocate_shared(nbytes if node_rank == 0 else 0, 1, comm=node_comm)
    buf, _ = win.Shared_query(0)
    raw = np.ndarray(buffer=buf, dtype=np.uint8, shape=(nbytes,))
    # Every process maps the segment at a page boundary, so one offset aligns it in all of them.
    offset = node_comm.bcast((-raw.ctypes.data) % SHM_ALIGN if node_rank == 0 else None, root=0)
    shm_view = raw[offset:offset + SHM_SLOTS * 8].view(np.uint64)
    if shm_view.ctypes.data % 64 != 0:
        raise RuntimeError("Shared window slot 0 is not 64-byte aligned on this rank.")
    win.Lock_all(MPI.MODE_NOCHECK)  # passive-target epoch so win.Sync() orders the shared loads/stores

    return {
        "comm": comm,
        "cart": cart,
        "rank": cart_rank,
        "coords": tuple(cart.Get_coords(cart_rank)),
        "nbrs": (nbr_up, nbr_down, nbr_left, nbr_right),
        "node_comm": node_comm,
        "shm_win": win,
        "shm_view": shm_view,
        "gpu_uuid": gpu_uuid,
        "cpu_sig": cpu_sig,
        "cpu_model": read_cpu_model(),
        "line_bytes": line_bytes,
        "seed": instance_seed,
    }


# ==============================================================================
# TASK 2: Rank-Local GPU Subdomain & Halo Pack/Unpack Kernels
# ==============================================================================
def initialize_fields(ctx):
    row_rank, col_rank = ctx["coords"]
    # Same seed on every rank -> every rank draws identical pulse parameters -> one consistent global field.
    rng = np.random.default_rng(ctx["seed"])
    n_pulses = 3
    pulse_x = rng.uniform(0.2, 0.8, n_pulses)
    pulse_y = rng.uniform(0.2, 0.8, n_pulses)
    pulse_amp = rng.uniform(1.0, 2.0, n_pulses)
    pulse_sigma = rng.uniform(0.04, 0.08, n_pulses)
    phase_x, phase_y = rng.uniform(0.0, 2.0 * np.pi, 2)
    src_x, src_y = rng.uniform(0.3, 0.7, 2)

    # Local index l = 0..257 is global index rank_coord * 256 + l, so x = gj * dx covers [0, 1].
    y = (row_rank * NY_LOCAL + np.arange(NY_LOCAL + 2)) * DY
    x = (col_rank * NX_LOCAL + np.arange(NX_LOCAL + 2)) * DX
    X, Y = np.meshgrid(x, y)  # (258, 258): [i, j] -> (y_i, x_j)

    kappa = 0.16 + 0.08 * np.sin(2.0 * np.pi * X + phase_x) * np.cos(2.0 * np.pi * Y + phase_y)
    u = np.zeros_like(X)
    for cx, cy, amp, sig in zip(pulse_x, pulse_y, pulse_amp, pulse_sigma):
        u += amp * np.exp(-((X - cx) ** 2 + (Y - cy) ** 2) / (2.0 * sig * sig))
    src = 0.05 * np.exp(-((X - src_x) ** 2 + (Y - src_y) ** 2) / (2.0 * 0.03**2))

    nbr_up, nbr_down, nbr_left, nbr_right = ctx["nbrs"]
    if nbr_up == MPI.PROC_NULL:
        u[0, :] = 0.0
    if nbr_down == MPI.PROC_NULL:
        u[-1, :] = 0.0
    if nbr_left == MPI.PROC_NULL:
        u[:, 0] = 0.0
    if nbr_right == MPI.PROC_NULL:
        u[:, -1] = 0.0
    return u, kappa, src


@cuda.jit
def pack_halos_kernel(u, send, n):
    k = cuda.grid(1)
    if k < n:
        send[UP * n + k] = u[1, k + 1]  # first owned row
        send[DOWN * n + k] = u[n, k + 1]  # last owned row
        send[LEFT * n + k] = u[k + 1, 1]  # first owned column (stride 258 doubles)
        send[RIGHT * n + k] = u[k + 1, n]  # last owned column


@cuda.jit
def unpack_halos_kernel(u, recv, n):
    k = cuda.grid(1)
    if k < n:
        u[0, k + 1] = recv[UP * n + k]
        u[n + 1, k + 1] = recv[DOWN * n + k]
        u[k + 1, 0] = recv[LEFT * n + k]
        u[k + 1, n + 1] = recv[RIGHT * n + k]


def exchange_halos_async(st, u, stream):
    """Pack on the GPU -> async D2H into pinned memory -> post tag-matched Irecv/Isend."""
    n = st.n
    pack_halos_kernel[st.pack_blocks, PACK_THREADS, stream](u, st.d_send, n)
    st.d_send.copy_to_host(st.h_send, stream=stream)
    stream.synchronize()  # MPI reads h_send from the host, so the D2H DMA must have landed
    reqs = [
        st.cart.Irecv([st.h_recv[d * n:(d + 1) * n], MPI.DOUBLE], source=st.nbrs[d], tag=OPPOSITE[d])
        for d in range(4)
    ]
    reqs += [
        st.cart.Isend([st.h_send[d * n:(d + 1) * n], MPI.DOUBLE], dest=st.nbrs[d], tag=d)
        for d in range(4)
    ]
    return reqs


def complete_halos(st, u, reqs, stream):
    """Waitall -> async H2D from pinned memory -> GPU unpack into ghosts 0 and 257.

    A PROC_NULL receive never touches its zero-filled segment, so wall ghosts are re-written with the
    Dirichlet value u = 0 without any boundary branch.
    """
    MPI.Request.Waitall(reqs)
    st.d_recv.copy_to_device(st.h_recv, stream=stream)
    unpack_halos_kernel[st.pack_blocks, PACK_THREADS, stream](u, st.d_recv, st.n)


# ==============================================================================
# TASK 3: 2D Shared-Memory Tiled Stencil & Dual-Stream Pipeline
# ==============================================================================
@cuda.jit
def tiled_stencil_kernel(u, u_new, kappa, src, r_start, r_end, c_start, c_end, dt, inv_dx2, inv_dy2, lam):
    tile = cuda.shared.array(shape=(SHARED_DIM, SHARED_DIM), dtype=float64)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    i = r_start + cuda.blockIdx.y * TILE_DIM + ty
    j = c_start + cuda.blockIdx.x * TILE_DIM + tx
    sy = ty + 1
    sx = tx + 1

    # Stage only the box plus its 1-cell apron: the deep-interior launch then never reads ghost
    # cells, which stream_boundary may be unpacking at the same moment.
    row_ok = i <= r_end + 1
    col_ok = j <= c_end + 1
    if row_ok and col_ok:
        tile[sy, sx] = u[i, j]
        if ty == 0:
            tile[0, sx] = u[i - 1, j]
        if tx == 0:
            tile[sy, 0] = u[i, j - 1]
    if ty == TILE_DIM - 1 and i <= r_end and col_ok:
        tile[sy + 1, sx] = u[i + 1, j]
    if tx == TILE_DIM - 1 and j <= c_end and row_ok:
        tile[sy, sx + 1] = u[i, j + 1]
    cuda.syncthreads()

    if i <= r_end and j <= c_end:
        uc = tile[sy, sx]
        kc = kappa[i, j]
        div_y = (0.5 * (kappa[i + 1, j] + kc) * (tile[sy + 1, sx] - uc)
                 - 0.5 * (kappa[i - 1, j] + kc) * (uc - tile[sy - 1, sx])) * inv_dy2
        div_x = (0.5 * (kappa[i, j + 1] + kc) * (tile[sy, sx + 1] - uc)
                 - 0.5 * (kappa[i, j - 1] + kc) * (uc - tile[sy, sx - 1])) * inv_dx2
        u_new[i, j] = uc + dt * (div_x + div_y - lam * uc * uc * uc + src[i, j])


def launch_stencil(st, u, u_new, box, stream):
    r_start, r_end, c_start, c_end = box
    grid = ((c_end - c_start + TILE_DIM) // TILE_DIM, (r_end - r_start + TILE_DIM) // TILE_DIM)
    tiled_stencil_kernel[grid, (TILE_DIM, TILE_DIM), stream](
        u, u_new, st.kappa, st.src, r_start, r_end, c_start, c_end, st.dt, st.inv_dx2, st.inv_dy2, LAMBDA_DAMP
    )


def step_serialized(st, u, u_new):
    """Reference: full halo exchange first, then one stencil over all owned cells, single stream."""
    reqs = exchange_halos_async(st, u, st.stream_boundary)
    complete_halos(st, u, reqs, st.stream_boundary)
    launch_stencil(st, u, u_new, st.full_box, st.stream_boundary)
    st.stream_boundary.synchronize()


def step_overlapped(st, u, u_new):
    reqs = exchange_halos_async(st, u, st.stream_boundary)  # pack + D2H + post Irecv/Isend
    launch_stencil(st, u, u_new, st.interior_box, st.stream_interior)  # [2..255]^2 needs no ghosts
    complete_halos(st, u, reqs, st.stream_boundary)  # Waitall + H2D + unpack
    for box in st.ring_boxes:
        launch_stencil(st, u, u_new, box, st.stream_boundary)
    st.stream_interior.synchronize()
    st.stream_boundary.synchronize()


def global_diagnostics(cart, u, u_prev):
    """Global energy sum(u)*dx*dy and L2 residual ||u^n - u^(n-1)|| via cart_comm.allreduce."""
    a = u.copy_to_host()[1:-1, 1:-1]
    b = u_prev.copy_to_host()[1:-1, 1:-1]
    energy = cart.allreduce(float(a.sum()) * DX * DY, op=MPI.SUM)
    res2 = cart.allreduce(float(((a - b) ** 2).sum()) * DX * DY, op=MPI.SUM)
    return energy, math.sqrt(res2)


def run_mode(st, step_fn, steps):
    u = cuda.to_device(st.u0)
    u_new = cuda.to_device(st.u0)  # wall ghosts start at 0 in both buffers; the stencil never writes ghosts
    trace = [(0, *global_diagnostics(st.cart, u, u_new))]
    start, stop = cuda.event(), cuda.event()
    st.cart.Barrier()
    start.record(st.stream_boundary)
    for n in range(1, steps + 1):
        step_fn(st, u, u_new)
        u, u_new = u_new, u
        if n % REPORT_EVERY == 0:
            trace.append((n, *global_diagnostics(st.cart, u, u_new)))
    stop.record(st.stream_boundary)
    stop.synchronize()
    ms = st.cart.allreduce(cuda.event_elapsed_time(start, stop), op=MPI.MAX)
    return u.copy_to_host(), ms, trace


def run_gpu_stencil(ctx) -> dict:
    cart = ctx["cart"]
    u0, kappa, src = initialize_fields(ctx)
    kappa_max = cart.allreduce(float(kappa[1:-1, 1:-1].max()), op=MPI.MAX)
    kappa_min = cart.allreduce(float(kappa[1:-1, 1:-1].min()), op=MPI.MIN)
    dt_max = DX * DX / (4.0 * kappa_max)
    dt = CFL_SAFETY * dt_max

    n = NX_LOCAL
    st = SimpleNamespace(
        cart=cart,
        nbrs=ctx["nbrs"],
        n=n,
        u0=u0,
        dt=dt,
        inv_dx2=1.0 / (DX * DX),
        inv_dy2=1.0 / (DY * DY),
        kappa=cuda.to_device(kappa),
        src=cuda.to_device(src),
        d_send=cuda.device_array(4 * n, dtype=np.float64),
        d_recv=cuda.device_array(4 * n, dtype=np.float64),
        h_send=cuda.pinned_array(4 * n, dtype=np.float64),
        h_recv=cuda.pinned_array(4 * n, dtype=np.float64),
        stream_boundary=cuda.stream(),
        stream_interior=cuda.stream(),
        pack_blocks=(n + PACK_THREADS - 1) // PACK_THREADS,
        full_box=(1, n, 1, n),
        interior_box=(2, n - 1, 2, n - 1),
        ring_boxes=((1, 1, 1, n), (n, n, 1, n), (2, n - 1, 1, 1), (2, n - 1, n, n)),
    )
    st.h_send[:] = 0.0
    st.h_recv[:] = 0.0  # pinned memory starts uninitialised; PROC_NULL segments must read as u = 0

    run_mode(st, step_serialized, 1)  # JIT warm-up so compilation is not timed
    run_mode(st, step_overlapped, 1)
    ser_u, ser_ms, ser_trace = run_mode(st, step_serialized, STENCIL_STEPS)
    ovl_u, ovl_ms, ovl_trace = run_mode(st, step_overlapped, STENCIL_STEPS)
    parity = cart.allreduce(float(np.abs(ser_u - ovl_u).max()), op=MPI.MAX)
    energies = [e for _, e, _ in ovl_trace]
    return {
        "kappa_min": kappa_min,
        "kappa_max": kappa_max,
        "dt_max_cfl": dt_max,
        "dt": dt,
        "ser_trace": ser_trace,
        "ovl_trace": ovl_trace,
        "gpu_serialized_ms": ser_ms,
        "gpu_overlapped_ms": ovl_ms,
        "parity_max_diff": parity,
        "energy_monotonic_decrease": all(b < a for a, b in zip(energies, energies[1:])),
    }


# ==============================================================================
# TASK 4: Multi-Rank Host Cache-Line False Sharing Sweep (MPI-3 shared window)
# ==============================================================================
@njit(nogil=True)
def host_shm_accumulate(shm_view, slot, iters, seed):
    inc = uint64(seed % 7 + 1)
    for _ in range(iters):
        v = shm_view[slot]
        # v never reaches 2**63, so (v >> 63) is 0 at run time, but LLVM cannot prove it. The store
        # address therefore depends on the loaded value, which blocks register promotion (LICM):
        # every iteration really loads and stores shm_view[slot] in the shared window.
        shm_view[slot + np.int64(v >> uint64(63))] = v + inc


def shm_fence(ctx):
    ctx["shm_win"].Sync()
    ctx["node_comm"].Barrier()
    ctx["shm_win"].Sync()


def benchmark_shm_strides(ctx) -> dict:
    nc, view = ctx["node_comm"], ctx["shm_view"]
    r, size = nc.Get_rank(), nc.Get_size()
    seed = (ctx["seed"] + r * 0x9E3779B97F4A7C15) % (1 << 63)  # < 2**63 so Numba types it as int64
    expected = SHM_ITERS * (seed % 7 + 1)

    cores = physical_core_cpus()
    pinned = nc.allreduce(len(cores) >= size, op=MPI.LAND)
    if pinned:
        os.sched_setaffinity(0, {cores[r]})  # one rank per physical core: SMT siblings would share an L1

    host_shm_accumulate(view, SHM_SLOTS - 8 + r, 10, seed)  # JIT warm-up on a scratch slot
    results = {}
    for stride in SHM_STRIDES:
        slot = r * stride
        runs, all_ok = [], True
        for _ in range(SHM_REPEATS):
            if r == 0:
                view[:] = 0
            shm_fence(ctx)
            t0 = time.perf_counter_ns()  # CLOCK_MONOTONIC: one clock shared by all ranks on the node
            host_shm_accumulate(view, slot, SHM_ITERS, seed)
            t1 = time.perf_counter_ns()
            shm_fence(ctx)
            ok = int(view[slot]) == expected
            all_ok = nc.allreduce(ok, op=MPI.LAND) and all_ok
            runs.append(nc.allreduce(t1, op=MPI.MAX) - nc.allreduce(t0, op=MPI.MIN))
        med = int(median(runs))
        results[stride] = {
            "slot_byte_offsets": [8 * k * stride for k in range(size)],
            "cache_lines_64B": [8 * k * stride // 64 for k in range(size)],
            "line_pairs_128B": [8 * k * stride // 128 for k in range(size)],
            "runs_ns": runs,
            "median_ns": med,
            "ns_per_update": med / SHM_ITERS,
            "counters_ok": all_ok,
        }
    return {
        "window_slots": SHM_SLOTS,
        "alignment_bytes": SHM_ALIGN,
        "iters_per_rank": SHM_ITERS,
        "repeats": SHM_REPEATS,
        "pinned_to_physical_cores": bool(pinned),
        "physical_cores_available": len(cores),
        "strides": results,
        "slowdown_stride1_vs_stride8": results[1]["median_ns"] / results[8]["median_ns"],
        "ratio_stride16_vs_stride8": results[16]["median_ns"] / results[8]["median_ns"],
    }


# ==============================================================================
# TASK 5: Cross-Tier Verification & Manifest Export
# ==============================================================================
def ptx_audit():
    asm = tiled_stencil_kernel.inspect_asm()
    ptx = "\n".join(asm.values()) if isinstance(asm, dict) else str(asm)
    counts = {op: ptx.count(op) for op in ("st.shared.f64", "ld.shared.f64", "bar.sync")}
    return counts, all(c > 0 for c in counts.values())


def llvm_audit(dispatcher) -> dict:
    ir = "\n".join(dispatcher.inspect_llvm().values())
    functions = re.split(r"\n(?=define )", ir)
    name = dispatcher.py_func.__name__
    # the native function, not Numba's "cpython" wrapper that boxes arguments
    core = [f for f in functions if f.startswith("define") and name in f.split("(")[0]
            and "cpython" not in f.split("(")[0]]
    body = "\n".join(core)
    return {
        "native_functions": len(core),
        "store_i64": len(re.findall(r"\bstore i64\b", body)),
        "load_i64": len(re.findall(r"= load i64\b", body)),
        "python_api_calls": len(re.findall(r"@_?Py\w+", body)),
    }


def main() -> int:
    if len(sys.argv) != 2:
        raise SystemExit("usage: mpirun -np 4 python3 hybrid_mpi_cuda_cache.py <github_username>")
    github_user = sys.argv[1]
    ctx = init_hybrid_context(github_user)
    cart, rank = ctx["cart"], ctx["rank"]

    if rank == 0:
        dev = cuda.get_current_device()
        print("=" * 78)
        print(f"GitHub user      : {github_user}")
        print(f"GPU UUID         : {ctx['gpu_uuid']}")
        print(f"GPU              : CC {dev.compute_capability}, {dev.MULTIPROCESSOR_COUNT} SMs, "
              f"{dev.ASYNC_ENGINE_COUNT} async copy engines")
        print(f"CPU              : {ctx['cpu_model']} | cpu_sig={ctx['cpu_sig']} | "
              f"{len(physical_core_cpus())} physical cores | line {ctx['line_bytes']} B")
        print(f"INSTANCE_SEED    : {ctx['seed']} (0x{ctx['seed']:016x})")
        print(f"MPI ranks        : {cart.Get_size()} in a {PROC_DIMS[0]}x{PROC_DIMS[1]} Cartesian grid, "
              f"periods=[False, False], MPI.PROC_NULL={MPI.PROC_NULL}, node_comm size={ctx['node_comm'].Get_size()}")
        print("=" * 78, flush=True)
    for r in range(cart.Get_size()):
        if r == rank:
            up, down, left, right = ctx["nbrs"]
            print(f"[rank {rank}] coords={ctx['coords']} up={up} down={down} left={left} right={right} "
                  f"local=(258, 258) shm slot0 addr mod 64 = {ctx['shm_view'].ctypes.data % 64}", flush=True)
        cart.Barrier()

    gpu = run_gpu_stencil(ctx)
    shm = benchmark_shm_strides(ctx)
    ptx_counts, ptx_ok = ptx_audit()
    llvm = llvm_audit(host_shm_accumulate)
    llvm_ok = llvm["native_functions"] >= 1 and llvm["store_i64"] >= 1 and llvm["python_api_calls"] == 0
    parity_ok = gpu["parity_max_diff"] < 1e-12
    shm_ok = all(s["counters_ok"] for s in shm["strides"].values())
    passed = parity_ok and ptx_ok and llvm_ok and shm_ok

    if rank == 0:
        print(f"\ndx = dy = 1/{GLOBAL_NX + 1} = {DX:.9e} | kappa range [{gpu['kappa_min']:.6f}, {gpu['kappa_max']:.6f}]")
        print(f"dt_max (CFL) = dx^2/(4 kappa_max) = {gpu['dt_max_cfl']:.6e} | dt = {CFL_SAFETY} * dt_max = {gpu['dt']:.6e}")
        for name, trace in (("serialized", gpu["ser_trace"]), ("overlapped", gpu["ovl_trace"])):
            print(f"\ncart_comm.allreduce diagnostics ({name}):")
            print(f"{'step':>6} {'energy E':>22} {'L2 residual':>22}")
            for s, e, res in trace:
                print(f"{s:>6} {e:>22.15e} {res:>22.15e}")
        g_ser, g_ovl = gpu["gpu_serialized_ms"], gpu["gpu_overlapped_ms"]
        print(f"\ngpu_serialized_ms : {g_ser:.3f} ({g_ser / STENCIL_STEPS:.4f} ms/step)")
        print(f"gpu_overlapped_ms : {g_ovl:.3f} ({g_ovl / STENCIL_STEPS:.4f} ms/step) -> {g_ser / g_ovl:.3f}x")
        print(f"parity_max_diff   : {gpu['parity_max_diff']:.3e} -> {'PASS' if parity_ok else 'FAIL'} (< 1e-12)")
        print(f"PTX audit         : {ptx_counts} -> {'PASS' if ptx_ok else 'FAIL'}")

        print(f"\nMPI-3 shared window: {SHM_SLOTS} uint64 slots, {SHM_ALIGN}-B aligned, "
              f"{SHM_ITERS:,} updates/rank, median of {SHM_REPEATS}, "
              f"pinned to physical cores: {shm['pinned_to_physical_cores']}")
        print(f"{'stride':>6} {'64B lines':>14} {'median ns':>16} {'ns/update':>10} {'vs s=8':>8} counters")
        s8 = shm["strides"][8]["median_ns"]
        for stride, s in shm["strides"].items():
            print(f"{stride:>6} {str(s['cache_lines_64B']):>14} {s['median_ns']:>16,} {s['ns_per_update']:>10.3f} "
                  f"{s['median_ns'] / s8:>8.2f} {'OK' if s['counters_ok'] else 'FAIL'}")
        print(f"slowdown stride=1 vs stride=8 : {shm['slowdown_stride1_vs_stride8']:.2f}x")
        print(f"stride=16 / stride=8          : {shm['ratio_stride16_vs_stride8']:.3f}")
        print(f"LLVM audit (host_shm_accumulate): {llvm} -> {'PASS' if llvm_ok else 'FAIL'}")

        manifest = {
            "github_username": github_user,
            "gpu_uuid": ctx["gpu_uuid"],
            "cpu_sig": ctx["cpu_sig"],
            "cpu_model": ctx["cpu_model"],
            "cpu_line_bytes": ctx["line_bytes"],
            "instance_seed": ctx["seed"],
            "mpi_ranks": cart.Get_size(),
            "cart_dims": PROC_DIMS,
            "cart_periods": [False, False],
            "node_comm_size": ctx["node_comm"].Get_size(),
            "global_interior_shape": [GLOBAL_NY, GLOBAL_NX],
            "local_padded_shape": [NY_LOCAL + 2, NX_LOCAL + 2],
            "stencil_steps": STENCIL_STEPS,
            "dx": DX,
            "dt": gpu["dt"],
            "kappa_min": gpu["kappa_min"],
            "kappa_max": gpu["kappa_max"],
            "dt_max_cfl": gpu["dt_max_cfl"],
            "lambda_damp": LAMBDA_DAMP,
            "energy_trace": [{"step": s, "energy": e, "l2_residual": res} for s, e, res in gpu["ovl_trace"]],
            "energy_monotonic_decrease": gpu["energy_monotonic_decrease"],
            "gpu_serialized_ms": gpu["gpu_serialized_ms"],
            "gpu_overlapped_ms": gpu["gpu_overlapped_ms"],
            "gpu_overlap_speedup": gpu["gpu_serialized_ms"] / gpu["gpu_overlapped_ms"],
            "parity_max_diff": gpu["parity_max_diff"],
            "parity_passed": parity_ok,
            "ptx_counts": ptx_counts,
            "ptx_audit_passed": ptx_ok,
            "shm_benchmark": {**shm, "strides": {str(k): v for k, v in shm["strides"].items()}},
            "llvm_audit": llvm,
            "llvm_audit_passed": llvm_ok,
            "verification_passed": passed,
            "generated_utc": datetime.now(timezone.utc).isoformat(),
        }
        with open("hybrid_lab_manifest.json", "w", encoding="utf-8") as f:
            json.dump(manifest, f, indent=2)
        print(f"\nverification_passed: {str(passed).lower()} -> wrote hybrid_lab_manifest.json", flush=True)

    ctx["shm_win"].Unlock_all()
    ctx["shm_win"].Free()
    return 0 if passed else 1


if __name__ == "__main__":
    sys.exit(main())


Writing hybrid_mpi_cuda_cache.py


### Cell 4 — Multi-rank hybrid execution

In [4]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "hybrid_mpi_cuda_cache.py",
    GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"Hybrid MPI-CUDA execution failed with code {proc.returncode}")

GitHub user      : 230103278
GPU UUID         : GPU-f3b1c3b2-e7b4-4a25-2908-82dc3311f7e2
GPU              : CC ComputeCapability(major=7, minor=5), 40 SMs, 3 async copy engines
CPU              : Intel(R) Xeon(R) CPU @ 2.00GHz | cpu_sig=x86_64::2 | 1 physical cores | line 64 B
INSTANCE_SEED    : 15170688048276696515 (0xd2891c66f3cbedc3)
MPI ranks        : 4 in a 2x2 Cartesian grid, periods=[False, False], MPI.PROC_NULL=-2, node_comm size=4
[rank 0] coords=(0, 0) up=-2 down=2 left=-2 right=1 local=(258, 258) shm slot0 addr mod 64 = 0
[rank 1] coords=(0, 1) up=-2 down=3 left=0 right=-2 local=(258, 258) shm slot0 addr mod 64 = 0
[rank 2] coords=(1, 0) up=0 down=-2 left=-2 right=3 local=(258, 258) shm slot0 addr mod 64 = 0
[rank 3] coords=(1, 1) up=1 down=-2 left=2 right=-2 local=(258, 258) shm slot0 addr mod 64 = 0

dx = dy = 1/513 = 1.949317739e-03 | kappa range [0.080001, 0.240000]
dt_max (CFL) = dx^2/(4 kappa_max) = 3.958168e-06 | dt = 0.8 * dt_max = 3.166534e-06

cart_comm.allreduce d

### Generated manifest

In [5]:
!cat hybrid_lab_manifest.json

{
  "github_username": "230103278",
  "gpu_uuid": "GPU-f3b1c3b2-e7b4-4a25-2908-82dc3311f7e2",
  "cpu_sig": "x86_64::2",
  "cpu_model": "Intel(R) Xeon(R) CPU @ 2.00GHz",
  "cpu_line_bytes": 64,
  "instance_seed": 15170688048276696515,
  "mpi_ranks": 4,
  "cart_dims": [
    2,
    2
  ],
  "cart_periods": [
    false,
    false
  ],
  "node_comm_size": 4,
  "global_interior_shape": [
    512,
    512
  ],
  "local_padded_shape": [
    258,
    258
  ],
  "stencil_steps": 300,
  "dx": 0.001949317738791423,
  "dt": 3.1665344448775936e-06,
  "kappa_min": 0.0800010372358577,
  "kappa_max": 0.23999989344273776,
  "dt_max_cfl": 3.958168056096992e-06,
  "lambda_damp": 0.15,
  "energy_trace": [
    {
      "step": 0,
      "energy": 0.12321167880449913,
      "l2_residual": 0.0
    },
    {
      "step": 100,
      "energy": 0.12317020338785876,
      "l2_residual": 3.7608727669738054e-05
    },
    {
      "step": 200,
      "energy": 0.12314035187457648,
      "l2_residual": 3.652179381756274e

### Cell 5 — Forensic engineering responses

### Analytical & Forensic Engineering Responses

> All measured values below come from this notebook's Colab run (Tesla T4 `GPU-f3b1c3b2-e7b4-4a25-2908-82dc3311f7e2`, `Intel(R) Xeon(R) CPU @ 2.00GHz`, `INSTANCE_SEED` = 15170688048276696515) and its `hybrid_lab_manifest.json` (`verification_passed: true`).

- **Question 1 (GPU Coalescing vs. CPU Cache-Line Granularity):**
  The local field is a C-contiguous `(258, 258)` float64 array, so a row is 258 × 8 = **2,064 B** and the halo column `u[1:257, 1]` is 256 doubles spaced 2,064 B apart.
  - **On the host CPU** the unit of transfer and coherence is the 64-byte line. Every column element lies in a different line, so reading 8 useful bytes moves 64 B from memory: **8× amplification**, with 256 lines (16,384 B) touched for 2,048 B of data. The stride prefetcher can hide the latency but not the wasted bandwidth.
  - **On the T4** the L1 line is 128 B, but DRAM and L2 are accessed in **32-byte sectors**. A warp reading 32 column elements issues 32 different addresses 2,064 B apart. That is **32 sectors and 1,024 B fetched for 256 useful bytes**: 25 % sector efficiency, or 4× amplification, and 32 separate L1 lines. Reading a row needs only 8 sectors (256 B, 100 %).
  - The CPU wastes 7/8 of every line, and the GPU wastes 3/4 of every sector.
  - **What `pack_halos_kernel` avoids:** it pays that 4× penalty once, on the device and over only 256 elements. It then writes all four strips coalesced into one contiguous 8,192 B buffer, which crosses PCIe as **one D2H DMA and one H2D DMA**.
  - **The bounding-box alternative** is a single contiguous DMA covering `u[1:257, 1]`. It would have to span 255·258 + 1 = 65,791 doubles = **526,328 B** for 2,048 useful bytes, i.e. **524,280 wasted bytes per column**, 257×. Each rank in the 2 × 2 grid sends one column and receives one per step, so that is **1,048,560 extra bytes per rank per step**. Over 4 ranks × 300 steps this comes to **1,258,272,000 B ≈ 1.26 GB** of avoidable PCIe traffic. The packed scheme moves 16,384 B per rank per step, **19.7 MB** in total. A bounding-box H2D into the ghost column would also overwrite 255 rows of owned cells with stale host data, so it would be wrong as well as slow.

- **Question 2 (Shared-Memory Tile Synchronization & PTX Audit):**
  `tiled_stencil_kernel.inspect_asm()` emitted **5 × `st.shared.f64`**, **5 × `ld.shared.f64`** and **1 × `bar.sync`** (`ptx_audit_passed: true`).
  - The stores are the centre `tile[ty+1, tx+1]` plus the four conditional apron stores (top for `ty==0`, bottom for `ty==15`, left for `tx==0`, right for `tx==15`).
  - The loads are the centre value `uc`, loaded once and reused, plus the N/S/W/E neighbours.
  - The single `bar.sync 0` is the one `cuda.syncthreads()`.

  **The read-after-write hazard without the barrier.** A 16 × 16 block is 8 warps, and warp *w* owns tile rows `2w+1` and `2w+2`. Its stencil reads rows `2w` and `2w+3`, which warps *w−1* and *w+1* store. Warp 0 also needs the top apron (row 0) and warp 7 the bottom apron (row 17), and every warp needs the `tx==0/15` column apron written by lanes in other half-warps. The SM's warp schedulers issue warps independently. So warp *w* can execute its `ld.shared` before its neighbour has executed the matching `st.shared`, and it reads stale shared memory left by the previous block on that SM. The results become wrong and non-deterministic, and `parity_max_diff` would expose this. Under Volta+ independent thread scheduling, even dependencies between lanes of the same warp are not ordered without a barrier. `bar.sync` makes every shared store before it visible to all 256 threads after it.

- **Question 3 (Dual-Stream Compute-Communication Overlap on Shared GPU Silicon):**
  Measured: `gpu_serialized_ms` = **746.9 ms** (**2.490 ms/step**) and `gpu_overlapped_ms` = **1022.5 ms** (**3.408 ms/step**), a speedup of **0.730×**.

  **One shared T4.** The four ranks are four OS processes, each with its own CUDA context on that one GPU. Without MPS, kernels from different contexts never occupy the SMs at the same time: the GPU **time-slices** between contexts. Only the two streams inside one rank can actually overlap. All four contexts also feed their H2D/D2H copies into the same **3** copy engines of the T4 and the same PCIe link, so the DMA queues of the four ranks are interleaved and serialized in hardware. On top of that, every halo is staged through host memory (GPU → pinned host → MPI shared-memory copy → pinned host → GPU), because Colab's OpenMPI has no GPUDirect path between processes.

  **Why overlap does not pay here.** The overlapped step issues 9 GPU operations and 2 stream syncs per step (pack, D2H, interior, H2D, unpack, 4 ring kernels); the serialized step issues 5 and 1. Each launch costs tens of µs of Python/Numba host time, and the four ring kernels run 16 blocks with 15/16 of their threads idle. The work being hidden is tiny: the deep interior is 254² ≈ 64.5 k cells, a few µs on a T4, and each halo message is only 2 KB. The extra launches cost more than the overlap hides, so overlap shows **a loss: the overlapped pipeline is 37 % slower** (+276 ms over 300 steps). On Colab the four ranks also share only 2 vCPUs, so the extra host-side launches and syncs of the overlapped step compete for CPU time as well.

  **Four dedicated GPUs with GPUDirect RDMA.** Each rank owns all its SMs and copy engines (no context time-slicing). The NIC reads and writes GPU memory directly, so the host staging copies disappear. Subdomains at production size (e.g. 4096² per GPU) make interior compute take milliseconds against microseconds of halo transfer, and step time falls from *compute + comm* towards *max(compute, comm)*.

- **Question 4 (MPI-3 Shared-Memory MESI RFO Ping-Pong):**
  `MPI.Win.Allocate_shared` maps the same physical page into all four processes. Cache coherence works on **physical** lines, so four separate processes false-share exactly as threads would. Each rank ran 40,000,000 read-modify-write updates on `shm_view[rank * stride]`. Measured wall-clock time (median of 3, from the earliest start to the latest finish over all ranks):

  | stride | slot offsets | 64 B lines | median ns | ns / update / rank | vs stride 8 |
  |---|---|---|---|---|---|
  | 1 (8 B) | 0, 8, 16, 24 | 0, 0, 0, 0 | 375,251,410 | 9.381 | 0.633× |
  | 8 (64 B) | 0, 64, 128, 192 | 0, 1, 2, 3 | 592,690,077 | 14.817 | 1.00× |
  | 16 (128 B) | 0, 128, 256, 384 | 0, 2, 4, 6 | 537,351,425 | 13.434 | 0.907× |

  **False-sharing slowdown (stride 1 vs stride 8) on Colab: 0.633×**, i.e. stride 1 was *faster*. This is the opposite of textbook false sharing, and the manifest explains why. The Colab VM (`Intel(R) Xeon(R) CPU @ 2.00GHz`, `cpu_sig = x86_64::2`) exposes **2 logical CPUs on 1 physical core**, so `pinned_to_physical_cores = false`: the four ranks could not each get their own core. The two vCPUs are SMT siblings that **share one L1D**. A line written by both hardware threads simply stays in **M** in that single L1, so there is no second core to send an RFO to, no invalidation and no ping-pong. On top of that, 4 runnable processes on 2 vCPUs means the kernel time-slices them, so each rank runs only about half the time and the wall time measures CPU sharing, not coherence. Stride 1 touches one line instead of four, and its runs were the most stable (363.6–378.1 ms). The stride-8 and stride-16 runs scatter by ±10 % (532.9–642.8 ms and 537.0–654.0 ms), which is typical hypervisor noise. All counters were still exact (`counters_ok: true`), and the LLVM audit of `host_shm_accumulate` shows **5 `store i64` / 5 `load i64`** and **0 Python API calls**, so every update really hit the shared window.

  For contrast, the **same script on a machine with dedicated cores** (my AMD Ryzen 5 8645HS, 6 physical cores, ranks pinned one per core, `pinned_to_physical_cores = true`) measured stride 1 = 936,075,767 ns (23.402 ns/update), stride 8 = 25,344,498 ns (0.634 ns/update) and stride 16 = 23,429,030 ns (0.586 ns/update). That is a **36.93× false-sharing slowdown**: what the MESI trace below predicts once four separate L1 caches compete for one line. False sharing needs at least two physical cores, which a 2-vCPU Colab VM does not have.

  **MESI trace.** Rank 0 writes `shm_view[0]` and Rank 1 writes `shm_view[1]`. Both are in line *L* (bytes 0–63 of the window). Assume *L* is in state **M** in Core 0's L1.
  1. **Rank 1 stores to `shm_view[1]`.** The store retires into Core 1's store buffer, but it can commit only when Core 1 owns *L* in E or M. Core 1's copy is **I**, so it sends a **Read-For-Ownership**.
  2. **The probe.** The L3 / snoop filter probes Core 0, which forwards the dirty line and invalidates its own copy: **Core 0 M → I**. Core 1 receives exclusive ownership, **I → E**, and commits the store: **E → M**.
  3. **Rank 0 writes `shm_view[0]` again.** Its copy is **I**, so the same happens in reverse: **Core 1 M → I**, **Core 0 I → E → M**.
  4. **The extra transaction from the loop.** Our loop loads before it stores. A core that has lost *L* first misses on the load. The owner downgrades **M → S** (or **M → O** under AMD MOESI), and the store then needs a second transaction, the upgrade **S → M**, which invalidates the other copy.

  While each RFO is outstanding the store buffer cannot drain. x86-TSO commits stores in order, so younger stores pile up behind the head store. Speculative loads of *L* that are invalidated by a probe also force pipeline machine clears. That is where the extra nanoseconds per update come from.

- **Question 5 (128-Byte Spatial Prefetcher Pairing & Padding Trade-Offs):**
  Measured: stride 8 = **592,690,077 ns** (**14.817 ns/update**) against stride 16 = **537,351,425 ns** (**13.434 ns/update**), a ratio of **0.907**. The window base is 128-byte aligned. With stride 8, ranks {0, 1} share 128-byte pair 0 and ranks {2, 3} share pair 1 (`line_pairs_128B: [0, 0, 1, 1]`); with stride 16 each rank has its own pair (`[0, 1, 2, 3]`).

  **How the pairing works.** Intel's L2 adjacent-line ("spatial") prefetcher treats memory as 128-byte-aligned pairs of 64-byte lines. On a miss to the even line it also fetches the odd one, and vice versa. AMD's L2 up/down prefetcher similarly fetches line ±1.

  **When 64-byte padding still suffers.** Every time Core 0 misses on its private line 0 (first touch, eviction, or after a context switch), the prefetcher can also pull line 1, which Core 1 holds in **M**. That snoop downgrades Core 1's copy, and Core 1's next store needs another RFO, even though no byte is shared. This is a secondary invalidation. 128-byte padding puts every rank in its own pair and removes it. On Colab, stride 16 came out 9 % faster than stride 8. That gap is smaller than the run-to-run spread and, with all ranks on one core's L1, no adjacent-line snoops between cores can happen, so it should not be read as a prefetcher effect. On the dedicated-core reference machine, stride 16 was 0.924× the stride-8 time (7.6 % faster). That is consistent with secondary invalidations between the ranks that share a 128-byte pair, though I did not count snoops with `perf` to prove it.

  **Why excessive padding hurts.** Padding multiplies the memory footprint. An 8-byte counter padded to 128 B occupies 16× the space, and every padded counter brings in at least one whole line. With a few counters this is free: 4 × 128 B = 512 B. With per-thread or per-cell arrays, e.g. 100,000 counters, the footprint grows from 0.8 MB to 12.8 MB and spills out of L2 into L3/DRAM, causing capacity misses. Large power-of-two strides also map every element into the same few cache sets. A 32 KiB, 8-way L1 has 64 sets indexed by `(addr / 64) mod 64`, so a 128 B stride uses only the even sets and halves the effective capacity. A 4 KiB stride would put everything in one set and cause conflict misses after 8 lines. Pad only the variables that are written concurrently by different cores, and only to the real line or pair size.

- **Question 6 (CFL Stability & Distributed Energy Dissipation):**
  Δx = Δy = 1/513 = 1.949318 × 10⁻³, so Δx² = 3.799840 × 10⁻⁶. The manifest gives `kappa_max` = **0.2399999**, so Δt_max = Δx² / (4 κ_max) = **3.958168e-06**. The simulation uses Δt = 0.8 · Δt_max = **3.166534e-06**. That gives Δt / Δt_max = **0.8 < 1**, so the Courant condition holds, and the mesh Fourier number is r = κ_max Δt / Δx² = **0.2 ≤ 0.25**. In von Neumann terms the worst-case (checkerboard) amplification factor is g = 1 − 8r = **−0.6**, so |g| < 1 for every Fourier mode.

  Global energy E = Σ u ΔxΔy from `cart_comm.allreduce` (overlapped run; the serialized trace is identical):

  | step | E | L2 residual ‖uⁿ − uⁿ⁻¹‖ |
  |---|---|---|
  | 0 | 0.123211678804 | 0 |
  | 100 | 0.123170203388 | 3.760873e-05 |
  | 200 | 0.123140351875 | 3.652179e-05 |
  | 300 | 0.123111493526 | 3.550188e-05 |

  E decreases at every report (drops of 4.15e-05, 2.99e-05, 2.89e-05; -0.0813 % in total), and the residual shrinks as well: the solution is stable and dissipative, not oscillating. The flux-form diffusion only moves heat between cells, so E changes only through three terms. Outflow through the u = 0 Dirichlet walls is small, because the pulses are centred in [0.2, 0.8]². The cubic damping −λu³ (λ = 0.15) removes heat wherever u > 0 and dominates. The weak source, 0.05·exp(−r²/2·0.03²), adds only about 2.8 × 10⁻⁴ per unit time. `parity_max_diff` = **0.0** confirms that both pipelines produce the same field on all 4 ranks.
